# JiraEstimator — End-to-End Pipeline

Проходим весь флоу от `data/seed.csv` до инференса, анализа ошибок и обратной связи с переобучением.

**Порядок запуска:** запустите ячейки сверху вниз.  
**Рабочая директория:** корень репозитория (`/path/to/JiraEstimator`).  
**Данные:** `data/seed.csv` должен быть создан через `scripts/get_jira_issues.py` (NDA, не коммитится).

## Настройка окружения

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook must run from project root
assert Path("config.py").exists(), (
    "Run from project root: jupyter notebook --notebook-dir=/path/to/JiraEstimator"
)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

def run(cmd):
    """Run shell command, print output, assert success."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

print(f"Working directory: {os.getcwd()}")
print(f"Python: {sys.executable}")

## Шаг 1: Feature Engineering + разметка риска

`prepare_data.py` принимает `data/seed.csv` (ырой экспорт из Jira) и:
- переводит `timespent` из секунд в рабочие дни (`logged_days`)
- вычисляет структурные признаки (наличие описания, тип задачи, длина текста)
- парсит `region`, `subsystem`, `commitments` из полей Jira
- размечает `risk_class` по соотношению `logged_days / story_points`

→ `data/dataset.csv`

In [ ]:
import pandas as pd

seed = pd.read_csv("data/seed.csv")
print(f"seed.csv: {len(seed)} строк, колонки: {list(seed.columns)}")

In [ ]:
run("python scripts/prepare_data.py")

df = pd.read_csv("data/dataset.csv")
print(f"\ndataset.csv: {len(df)} строк")
df[['summary', 'logged_days', 'story_points', 'region', 'subsystem', 'commitments', 'risk_class']].head(5)

## Шаг 2: Стратифицированное разбиение 80/10/10

`split_data.py` делит данные с сохранением распределения `risk_class`.

→ `data/train.csv`, `data/val.csv`, `data/test.csv`

In [ ]:
run("python scripts/split_data.py")

for name in ["train", "val", "test"]:
    part = pd.read_csv(f"data/{name}.csv")
    print(f"{name:5s}: {len(part):4d} строк  "
          f"risk_class dist: {dict(part['risk_class'].value_counts().sort_index())}")

## Шаг 3: LSA-эмбеддинги

`extract_bm25_lsa.py` обучает TF-IDF (sublinear_tf, min_df=2, 25k vocab) + TruncatedSVD(32) на train-корпусе,
затем трансформирует train/val/test.

→ `embeddings/bm25_lsa_pipeline.pkl`, `embeddings/train_X.npy`, `val_X.npy`, `test_X.npy`

In [ ]:
run("python scripts/extract_bm25_lsa.py")

import numpy as np
for name in ["train", "val", "test"]:
    arr = np.load(f"embeddings/{name}_X.npy")
    print(f"{name}_X.npy: {arr.shape}")

## Шаг 4: Обучение моделей

### 4a. Регрессор усилий (CatBoostRegressor)
5-fold KFold, цель: `log1p(logged_days)`. Логирует метрики в MLflow.

→ `models/estimate_fold_0..4.cbm`

In [ ]:
run("python scripts/train_catboost_estimate.py")

### 4b. Классификатор риска (CatBoostClassifier)
5-fold KFold, 3 класса (Low/Medium/Critical). Логирует в MLflow.

→ `models/risk_fold_0..4.cbm`

In [ ]:
run("python scripts/train_catboost_risk.py")

## Шаг 5: Просмотр MLflow-экспериментов

Все запуски логируются в `mlruns/` (file-based, сервер не нужен).  
Для полного UI: `mlflow ui --backend-store-uri file:./mlruns --port 5000`

In [ ]:
import mlflow

client = mlflow.tracking.MlflowClient("file:./mlruns")
for exp in client.search_experiments():
    runs = client.search_runs(
        exp.experiment_id, order_by=["start_time DESC"], max_results=3
    )
    if not runs:
        continue
    print(f"\nExperiment: {exp.name}")
    for r in runs:
        m = r.data.metrics
        interesting = {k: round(v, 4) for k, v in m.items()
                       if k in ("test_r2", "test_mae", "val_r2", "test_accuracy", "cv_r2_mean")}
        print(f"  {r.info.run_id[:8]}  {interesting}")

## Шаг 6: Сравнение с наивными базовыми линиями

Базовые линии: глобальная медиана, медиана по подсистеме, медиана по региону.
Сравниваем R² с моделью (~0.144) для количественной оценки прироста.

In [ ]:
run("python scripts/compute_baseline.py")

## Шаг 7: Анализ ошибок

Смотрим, где модель ошибается больше всего: по подсистемам, регионам, отдельным тикетам.

In [ ]:
run("python scripts/analyze_errors.py --top 10")

## Шаг 8: Инференс

Встроенный пример: берём один тикет и прогоняем через ансамбль моделей без API.

In [ ]:
import pickle
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from config import CONFIG
from feature_engineering import compute_text_features

# Load LSA pipeline
with open("embeddings/bm25_lsa_pipeline.pkl", "rb") as f:
    lsa_pipeline = pickle.load(f)

# Load estimate ensemble
est_folds = []
for i in range(5):
    m = CatBoostRegressor()
    m.load_model(f"models/estimate_fold_{i}.cbm")
    est_folds.append(m)

# Load risk ensemble
risk_folds = []
for i in range(5):
    m = CatBoostClassifier()
    m.load_model(f"models/risk_fold_{i}.cbm")
    risk_folds.append(m)

print(f"Loaded {len(est_folds)} estimate folds + {len(risk_folds)} risk folds")

In [ ]:
ticket = {
    "summary": "Реализовать выгрузку СЭМД протокола осмотра врача",
    "description": "Добавить формирование CDA R2 по утверждённому шаблону. Шаблон согласован с главным врачом.",
    "region": "БАЗОВЫЙ",
    "subsystem": "СЭМД/Выгрузка",
    "commitments": "ТУР",
}

text = ticket["summary"] + " " + ticket["description"]
emb = lsa_pipeline.transform([text])          # (1, 32)
text_feats = compute_text_features(text, ticket["description"])

row = {**{f"emb_{i}": emb[0, i] for i in range(emb.shape[1])}, **text_feats,
       "region": ticket["region"], "subsystem": ticket["subsystem"], "commitments": ticket["commitments"]}
feat_df = pd.DataFrame([row])
cat_cols = ["region", "subsystem", "commitments"]

est_pool = Pool(feat_df, cat_features=cat_cols)
risk_pool = Pool(feat_df, cat_features=cat_cols)

# Estimate: ensemble mean
preds_days = [np.expm1(m.predict(est_pool)[0]) for m in est_folds]
mean_days = np.mean(preds_days)

# Risk: ensemble mean probabilities
risk_probs = np.mean([m.predict_proba(risk_pool)[0] for m in risk_folds], axis=0)
p_medium, p_critical = risk_probs[1], risk_probs[2]
adjusted_days = mean_days * (1 + 0.15 * p_medium + 0.50 * p_critical)

print(f"Тикет: {ticket['summary']}")
print(f"Базовая оценка: {mean_days:.2f} ФТЕ-дней ({mean_days*8:.1f} часов)")
print(f"Риск: Low={risk_probs[0]:.1%}  Medium={p_medium:.1%}  Critical={p_critical:.1%}")
print(f"Скорректированная (с риск-буфером): {adjusted_days:.2f} дней ({adjusted_days*8:.1f} часов)")

## Шаг 9: Петля обратной связи

Сидируем 20 синтетических фидбэков от команды МИС, проверяем счётчик до порога переобучения.

In [ ]:
# Очищаем старые синтетические записи и добавляем 20 новых
run("python scripts/seed_synthetic_feedback.py --clear")
run("python scripts/seed_synthetic_feedback.py")

In [ ]:
import sqlite3
from config import CONFIG

conn = sqlite3.connect(CONFIG["db_path"])
rows = conn.execute(
    "SELECT summary, predicted_days, actual_days, commitments FROM feedback ORDER BY id DESC LIMIT 5"
).fetchall()
unused = conn.execute("SELECT COUNT(*) FROM feedback WHERE is_used_for_training=0").fetchone()[0]
total  = conn.execute("SELECT COUNT(*) FROM feedback").fetchone()[0]
conn.close()

threshold = CONFIG.get("retrain_threshold", 50)
print(f"Feedback: {total} всего, {unused} неиспользованных")
print(f"Порог переобучения: {threshold}")
print(f"Статус: {'READY ✓' if unused >= threshold else f'Нужно ещё {threshold - unused}'}")
print()
print(f"{'Summary':<55} {'Pred':>6} {'Act':>6} {'Commit':>8}")
print("-" * 78)
for r in rows:
    print(f"{r[0][:54]:<55} {r[1]:>6.1f} {r[2]:>6.1f} {r[3]:>8}")

## Шаг 10: Переобучение по фидбэку

Пробный запуск — проверяем статус без реального переобучения.  
Для запуска реального переобучения (когда накоплено ≥50 фидбэков): `python scripts/retrain.py`

In [ ]:
run("python scripts/retrain.py --dry-run")

---

## Готово

Весь пайплайн пройден. Модели в `models/`, метрики в MLflow (`mlruns/`), API: `uvicorn app:app --port 8000`.

**Ссылки:**
- `scripts/` — отдельные этапы пайплайна
- `app.py` — FastAPI-сервер (predict / explain / feedback / metrics / health)
- `demo.py` — Streamlit-интерфейс